# Academic Dropout Risk Analysis: Predictive Modeling

**Objective:** build a classification model that identifies students at risk of dropping out using only information assumed to be available after the first semester (Scenario A).

This notebook follows `01_eda.ipynb`, which describes the data and the associations with dropout.

# 1. Libraries

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 2. Setup

In [2]:
df = pd.read_csv("../data/raw/student_dropout_risk_2026.csv")
print(f"Dataset shape: {df.shape}")

Dataset shape: (100000, 34)


# 3. Problem definition

- **Target:** `dropout` (1 = dropped out, 0 = stayed). About 32% of students dropped out.
- **Scenario A (early warning):** the model flags at-risk students using the information selected in section 4 as first-semester information.

# 4. Feature selection

**Scenario A (early-warning feature set).** The dataset description suggests an early-warning task using semester 1 features but does not define which variables belong to it, so the selection below is a design decision.

- **Not used as features:** `person_id` (identifier) and `dropout` (target).
- **Excluded because they are not first-semester information:** `gpa_semester_2` (measured after the second semester) and `semester`, `units_enrolled`, `units_passed`, `units_failed`, `missed_deadlines` (described as referring to the current semester; `units_passed` and `units_failed` are tied to `units_enrolled` by definition).
- **Included (26 features):** demographic, socioeconomic and institutional variables, admission and previous qualification grades, `gpa_semester_1`, and engagement and wellbeing variables.

**Assumption:** the dataset does not state the period covered by `attendance_rate`, `assignments_submitted_rate`, `library_visits_per_month`, `lms_logins_per_week`, `stress_level`, `satisfaction_score`, `tuition_up_to_date` and `mental_health_support`. They are assumed to be available after the first semester; this cannot be verified from the data.

A final comparison with all features will show how much the excluded variables add.

In [3]:
target = "dropout"
excluded = ["person_id", "gpa_semester_2", "semester",
            "units_enrolled", "units_passed", "units_failed", "missed_deadlines"]

features = [c for c in df.columns if c not in excluded + [target]]

# Split the selected features by type (needed to prepare the data)
cat_features = df[features].select_dtypes(exclude="number").columns.tolist()
num_features = df[features].select_dtypes(include="number").columns.tolist()

print("Number of features:", len(features))
print("Categorical:", cat_features)
print("Numeric:", num_features)

Number of features: 26
Categorical: ['gender', 'nationality', 'family_income_level', 'course_type', 'study_mode']
Numeric: ['age', 'first_generation', 'disability_status', 'scholarship_holder', 'tuition_up_to_date', 'part_time_job', 'weekly_work_hours', 'international_student', 'admission_grade', 'previous_qualification_grade', 'gpa_semester_1', 'attendance_rate', 'assignments_submitted_rate', 'library_visits_per_month', 'lms_logins_per_week', 'extracurricular_activities', 'stress_level', 'mental_health_support', 'satisfaction_score', 'commute_time_mins', 'sleep_hours']


# 5. Train / Test split

The data are split into 80% training and 20% test sets, stratified by `dropout` so both keep the same dropout rate (about 32%). The test set is not used until the final evaluation. Preprocessing (scaling and encoding) will be fitted on the training set only, inside a pipeline, to avoid information leakage from the test set.

In [5]:
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Dropout rate - train:", round(y_train.mean(), 4), "| test:", round(y_test.mean(), 4))

Train: (80000, 26) | Test: (20000, 26)
Dropout rate - train: 0.32 | test: 0.32
